In [1]:
import cv2
import numpy as np
import os, glob
from pathlib import Path
import pandas as pd

In [2]:
BASE_DIR   = r"D:\program vscode\MoneyLens\ai\Dataset_ocr"
TEMAN_DIR  = os.path.join(BASE_DIR, "Preprocessing Citra", "dataset-cleaning-noise")
OUT_DIR    = os.path.join(BASE_DIR, "preprocessed")
 
SPLITS     = ["train", "valid", "test"]
IMG_SIZE   = 640
IMG_H      = 32
IMG_W      = 128
CHANNELS   = 1
MIN_CROP   = 5
PADDING    = 6
 
CLASS_MAP = {
    0: "QTY",
    1: "harga_satuan",
    2: "nama_produk",
    3: "tanggal",
    4: "total_harga_barang",
    5: "total_transaksi",
}
 
PRIORITAS = ["total_transaksi", "tanggal"]

In [3]:
def read_yolo_labels(label_path: str, img_w: int, img_h: int) -> list:
    labels = []
    if not os.path.exists(label_path):
        return labels
 
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            try:
                cls_id = int(parts[0])
                cx     = float(parts[1]) * img_w
                cy     = float(parts[2]) * img_h
                w      = float(parts[3]) * img_w
                h      = float(parts[4]) * img_h
 
                x1 = int(cx - w / 2)
                y1 = int(cy - h / 2)
                x2 = int(cx + w / 2)
                y2 = int(cy + h / 2)
 
                labels.append({
                    "class_id"  : cls_id,
                    "class_name": CLASS_MAP.get(cls_id, f"unknown_{cls_id}"),
                    "x1": max(0, x1), "y1": max(0, y1),
                    "x2": min(img_w, x2), "y2": min(img_h, y2),
                })
            except Exception:
                continue
 
    return labels

In [4]:
def preprocess_crop(crop_cv: np.ndarray) -> np.ndarray:
    """
    Preprocessing untuk 1 crop region teks.
    Input sudah grayscale dari teman, jadi skip grayscale.
 
    Alur:
      1. Cek ukuran minimum
      2. Adaptive Threshold → binarisasi bersih
      3. Resize 128x32      → sesuai input model
      4. Normalisasi        → pixel [0.0, 1.0]
 
    Returns: np.ndarray shape (32, 128, 1), dtype float32
    """
    # 1. Cek ukuran minimum
    h, w = crop_cv.shape[:2]
    if h < MIN_CROP or w < MIN_CROP:
        raise ValueError(f"Crop terlalu kecil ({w}x{h}px)")
 
    if len(crop_cv.shape) == 3:
        gray = cv2.cvtColor(crop_cv, cv2.COLOR_BGR2GRAY)
    else:
        gray = crop_cv
 
    # 2. Adaptive Threshold — binarisasi
    # blockSize disesuaikan dengan ukuran crop
    block = max(3, min(h, w) // 2 * 2 + 1)
    binary = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, block, 10
    )
 
    # 3. Resize ke ukuran input model
    resized = cv2.resize(binary, (IMG_W, IMG_H),
                         interpolation=cv2.INTER_AREA)
 
    # 4. Normalisasi [0.0, 1.0] + channel dimension
    normalized = resized.astype(np.float32) / 255.0
    return normalized.reshape(IMG_H, IMG_W, CHANNELS)

In [5]:
def crop_with_padding(img: np.ndarray,
                      x1: int, y1: int,
                      x2: int, y2: int) -> np.ndarray:
    """Crop region dengan padding agar teks tidak terpotong"""
    ih, iw = img.shape[:2]
    x1p = max(0,  x1 - PADDING)
    y1p = max(0,  y1 - PADDING)
    x2p = min(iw, x2 + PADDING)
    y2p = min(ih, y2 + PADDING)
    return img[y1p:y2p, x1p:x2p]

In [6]:
def crop_by_class(img_cv: np.ndarray, df_ann: pd.DataFrame, filename: str) -> dict:
    """
    Crop tiap region teks berdasarkan kelas dari annotations CSV.
 
    Returns:
      dict { class_name: [crop_array, ...] }
    """
    result = {cls: [] for cls in LABEL_CLASSES}
 
    rows = df_ann[df_ann["filename"] == filename]
    for _, row in rows.iterrows():
        cls = str(row.get("class", ""))
        if cls not in LABEL_CLASSES:
            continue
        try:
            x1 = max(0, int(row["xmin"]))
            y1 = max(0, int(row["ymin"]))
            x2 = min(img_cv.shape[1], int(row["xmax"]))
            y2 = min(img_cv.shape[0], int(row["ymax"]))
 
            if x2 <= x1 or y2 <= y1:
                continue
 
            crop = add_padding(img_cv, img_cv, x1, y1, x2, y2)
            if crop.size > 0:
                result[cls].append(crop)
        except Exception:
            continue
 
    return result

In [7]:
def process_image(img_path: str, label_path: str,
                  crops_dir: str, arrays_dir: str) -> dict:
    """
    Proses 1 gambar:
      - Baca gambar + label YOLO
      - Crop per kelas + padding
      - Preprocessing → simpan .npy + .png
    """
    img_cv = cv2.imread(img_path)
    if img_cv is None:
        raise ValueError(f"Tidak bisa buka: {img_path}")
 
    ih, iw = img_cv.shape[:2]
    stem   = Path(img_path).stem
 
    # Baca label YOLO
    labels = read_yolo_labels(label_path, iw, ih)
 
    summary = {v: 0 for v in CLASS_MAP.values()}
    counts  = {}   # hitung index per kelas
 
    for lbl in labels:
        cls  = lbl["class_name"]
        idx  = counts.get(cls, 0)
        counts[cls] = idx + 1
 
        try:
            # Crop + padding
            crop = crop_with_padding(
                img_cv, lbl["x1"], lbl["y1"],
                lbl["x2"], lbl["y2"]
            )
 
            # Preprocessing
            arr = preprocess_crop(crop)
 
            # Simpan PNG preview
            png = f"{stem}_{cls}_{idx:02d}.png"
            cv2.imwrite(
                os.path.join(crops_dir, png),
                (arr[:, :, 0] * 255).astype(np.uint8)
            )
 
            # Simpan NPY untuk input model
            npy = f"{stem}_{cls}_{idx:02d}.npy"
            np.save(os.path.join(arrays_dir, npy), arr)
 
            summary[cls] = summary.get(cls, 0) + 1
 
        except ValueError as e:
            print(f"    [SKIP] {cls}: {e}")
        except Exception as e:
            print(f"    [ERROR] {cls}: {e}")
 
    return summary

In [8]:
print("=" * 60)
print("TASK 2: PREPROCESSING FORMAT INPUT MODEL OCR")
print("=" * 60)
print(f"Sumber      : dataset-cleaning-noise (hasil teman)")
print(f"Target size : {IMG_H}x{IMG_W}px | grayscale | float32")
print(f"Padding     : {PADDING}px tiap sisi")
print(f"Class map   : {CLASS_MAP}")
print()
 
grand_total = {v: 0 for v in CLASS_MAP.values()}
 
for split in SPLITS:
    img_dir    = os.path.join(TEMAN_DIR, split, "images")
    lbl_dir    = os.path.join(TEMAN_DIR, split, "labels")
    crops_dir  = os.path.join(OUT_DIR, split, "crops")
    arrays_dir = os.path.join(OUT_DIR, split, "arrays")
    os.makedirs(crops_dir,  exist_ok=True)
    os.makedirs(arrays_dir, exist_ok=True)
 
    if not os.path.exists(img_dir):
        print(f"[{split}] Folder tidak ditemukan: {img_dir}")
        continue
 
    img_paths = sorted(
        glob.glob(os.path.join(img_dir, "*.jpg")) +
        glob.glob(os.path.join(img_dir, "*.png"))
    )
 
    print(f"\n[{split.upper()}] {len(img_paths)} gambar")
    ok = fail = 0
 
    split_total = {v: 0 for v in CLASS_MAP.values()}
 
    for i, img_path in enumerate(img_paths, 1):
        stem       = Path(img_path).stem
        label_path = os.path.join(lbl_dir, stem + ".txt")
 
        try:
            summary = process_image(
                img_path, label_path, crops_dir, arrays_dir)
            for cls, n in summary.items():
                split_total[cls]  += n
                grand_total[cls]  += n
            found = {k: v for k, v in summary.items() if v > 0}
            print(f"  [{i:03d}/{len(img_paths):03d}] ✓ {stem[:40]} | {found}")
            ok += 1
        except Exception as e:
            print(f"  [{i:03d}/{len(img_paths):03d}] ✗ {stem[:40]} → {e}")
            fail += 1
 
    print(f"\n  Ringkasan {split}:")
    for cls, n in split_total.items():
        prio = " ← PRIORITAS" if cls in PRIORITAS else ""
        print(f"    {cls:<22}: {n:>4} crop{prio}")
    print(f"  Berhasil: {ok} | Gagal: {fail}")
 
print(f"\n{'='*60}")
print("TOTAL CROP PER KELAS (semua split)")
print(f"{'='*60}")
for cls, n in grand_total.items():
    prio = " ← PRIORITAS" if cls in PRIORITAS else ""
    print(f"  {cls:<22}: {n:>5} array .npy{prio}")
print(f"\nOutput: {OUT_DIR}")
print(f"{'='*60}")

TASK 2: PREPROCESSING FORMAT INPUT MODEL OCR
Sumber      : dataset-cleaning-noise (hasil teman)
Target size : 32x128px | grayscale | float32
Padding     : 6px tiap sisi
Class map   : {0: 'QTY', 1: 'harga_satuan', 2: 'nama_produk', 3: 'tanggal', 4: 'total_harga_barang', 5: 'total_transaksi'}


[TRAIN] 282 gambar
  [001/282] ✓ 10_4_26-nota-tinta-isolasi__jpg.rf.9ccf2 | {'QTY': 2, 'harga_satuan': 2, 'nama_produk': 2, 'tanggal': 1, 'total_harga_barang': 2, 'total_transaksi': 1}
  [002/282] ✓ 10_4_26-nota-trashbag__jpg.rf.45eca1cbce | {'QTY': 1, 'harga_satuan': 1, 'nama_produk': 1, 'tanggal': 1, 'total_harga_barang': 1, 'total_transaksi': 1}
  [003/282] ✓ 16_03_26-nota-hvs_jpg.rf.0025fea66265e3c | {'QTY': 1, 'nama_produk': 1, 'tanggal': 1, 'total_harga_barang': 1, 'total_transaksi': 1}
  [004/282] ✓ 16_4_26-nota-trashbag__jpg.rf.aa0788dfb9 | {'QTY': 1, 'harga_satuan': 1, 'nama_produk': 1, 'tanggal': 1, 'total_harga_barang': 1, 'total_transaksi': 1}
  [005/282] ✓ 20260507_155458_jpg.rf.0b96d